## 1. Objective

This notebook evaluates whether ENTSOG load **point forecasts** can be transformed into **predictive distributions** suitable for probabilistic validation:

- Probability Integral Transform (PIT) histogram + dependence (ACF)
- Central interval empirical coverage (80%, 90%)
- Conditioning / regularization strategies under limited history

Core question:

> Can a statistically defensible predictive distribution be constructed from point forecasts and historical residuals?

In [ ]:
from pathlib import Path
import os

# Repo root assumed to be the parent of /notebooks
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

# ---- Input ----
# NOTE: This notebook targets the 90-day sample. Adjust if your filename differs.
PATH = REPO_ROOT / "data" / "entsog_sample_90days.csv"

print("CWD:", Path.cwd())
print("Loading:", PATH, "exists?", PATH.exists())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

COL_T, COL_Y, COL_YHAT = "timestamp", "y", "y_hat"

## 2. Load + canonicalize columns
We standardize to canonical names: `timestamp`, `y`, `y_hat`, sort by time, and drop missing values.

In [ ]:
df = pd.read_csv(PATH)

# Clean header names (important if there are trailing spaces)
df.columns = [c.strip() for c in df.columns]
print("Columns:", df.columns.tolist())  # sanity check

# Rename to canonical names (handles a blank/Unnamed first column if present)
rename_map = {df.columns[0]: COL_T}  # first column -> timestamp

# common ENTSOG-like column names
if "Load" in df.columns:
    rename_map["Load"] = COL_Y
if "Load forecast" in df.columns:
    rename_map["Load forecast"] = COL_YHAT

df = df.rename(columns=rename_map)

# safety: if already canonical, do nothing
assert COL_T in df.columns, f"Missing {COL_T} after rename."
assert COL_Y in df.columns, f"Missing {COL_Y} after rename."
assert COL_YHAT in df.columns, f"Missing {COL_YHAT} after rename."

df[COL_T] = pd.to_datetime(df[COL_T])
df = df.sort_values(COL_T).dropna(subset=[COL_Y, COL_YHAT]).reset_index(drop=True)

df.head(), df.shape

## 3. Baseline A — Gaussian residuals with rolling volatility

Assume residuals are conditionally Normal with time-varying standard deviation estimated by a rolling window.

- Residual: `r_t = y_t - ŷ_t`
- Rolling volatility: `σ_t = std(r_{t-W+1:t})`
- PIT: `u_t = Φ(r_t / σ_t)`

In [ ]:
# rolling residual volatility model (baseline)
resid = df[COL_Y] - df[COL_YHAT]

# 7 days of quarter-hourly -> 7*96 = 672
W_target = 7 * 96
W = min(W_target, max(30, len(df)//3))   # at least 30 points, else 1/3 of data
print("Baseline-A window W:", W)

sigma = resid.rolling(W, min_periods=W).std()

# align: only evaluate when sigma exists
mask = sigma.notna()
y = df.loc[mask, COL_Y].to_numpy()
yhat = df.loc[mask, COL_YHAT].to_numpy()
sig = sigma.loc[mask].to_numpy()
t_eval = df.loc[mask, COL_T].to_numpy()

# PIT
z = (y - yhat) / sig
u = norm.cdf(z)

print("n_rows:", len(df))
print("sigma non-NaN:", int(mask.sum()))
print("PIT length:", len(u))
u[:5], u.shape

In [ ]:
# PIT histogram
plt.figure()
plt.hist(u, bins=20, range=(0, 1))
plt.title("PIT histogram (Baseline A: Gaussian residual + rolling σ)")
plt.xlabel("u")
plt.ylabel("count")
plt.show()

# PIT autocorrelation (lag 1..50)
def autocorr(x, lag):
    x = x - x.mean()
    return np.corrcoef(x[:-lag], x[lag:])[0,1]

max_lag = min(50, len(u) - 1)
acs = [autocorr(u, k) for k in range(1, max_lag + 1)]

plt.figure()
plt.plot(range(1, max_lag + 1), acs)
plt.title("PIT autocorrelation (Baseline A)")
plt.xlabel("Lag")
plt.ylabel("ACF")
plt.show()

In [ ]:
# coverage check for central intervals (80%, 90%)
for alpha in [0.2, 0.1]:
    lo = norm.ppf(alpha/2, loc=yhat, scale=sig)
    hi = norm.ppf(1-alpha/2, loc=yhat, scale=sig)
    cov = np.mean((y >= lo) & (y <= hi))
    print(f"Baseline A — nominal {int((1-alpha)*100)}%: empirical {cov:.3f}  (n={len(y)})")

## 4. Baseline B — Rolling empirical residual distribution (distribution-free)

Replace the Normal assumption with a rolling empirical CDF (ECDF) for residuals.

- PIT via ECDF rank within trailing window
- Central intervals via rolling residual quantiles added to `ŷ_t`

In [ ]:
# ---------- Baseline B: Rolling empirical residual distribution ----------
resid_all = (df[COL_Y] - df[COL_YHAT]).to_numpy()

# 15-min data: 96 ≈ 1 day. Auto-shrink if sample is smaller.
W = min(96, len(df) - 1)
if W < 30:
    raise ValueError(f"Too few rows ({len(df)}) for rolling residual window. Need >= 30-ish.")

uB = np.full(len(df), np.nan, dtype=float)

for t in range(W - 1, len(df)):
    window = resid_all[t - W + 1 : t + 1]
    r_t = resid_all[t]
    # ECDF with continuity correction to avoid 0/1 endpoints
    uB[t] = (np.sum(window < r_t) + 0.5 * np.sum(window == r_t)) / (len(window) + 1.0)

uB = uB[~np.isnan(uB)]
print("Baseline-B PIT length:", len(uB), "Window W:", W)

In [ ]:
plt.figure()
plt.hist(uB, bins=20, range=(0,1))
plt.title("PIT histogram (Baseline B: rolling empirical residuals)")
plt.xlabel("u")
plt.ylabel("count")
plt.show()

def autocorr(x, lag):
    x = x - x.mean()
    return np.corrcoef(x[:-lag], x[lag:])[0, 1]

max_lag = min(50, len(uB) - 1)
acsB = [autocorr(uB, k) for k in range(1, max_lag + 1)]

plt.figure()
plt.plot(range(1, max_lag + 1), acsB)
plt.title("PIT autocorrelation (Baseline B)")
plt.xlabel("Lag")
plt.ylabel("ACF")
plt.show()

In [ ]:
# Coverage check using rolling residual quantiles (central intervals)
y_all = df[COL_Y].to_numpy()
yhat_all = df[COL_YHAT].to_numpy()

for alpha in [0.2, 0.1]:
    covered = []
    for t in range(W - 1, len(df)):
        window = resid_all[t - W + 1 : t + 1]
        lo_r = np.quantile(window, alpha / 2)
        hi_r = np.quantile(window, 1 - alpha / 2)
        lo = yhat_all[t] + lo_r
        hi = yhat_all[t] + hi_r
        covered.append((y_all[t] >= lo) & (y_all[t] <= hi))
    cov = float(np.mean(covered))
    print(f"Baseline B — nominal {int((1-alpha)*100)}%: empirical {cov:.3f}  (n={len(covered)})")

## 5. Conditioning experiments (seasonality proxies)

### 5.1 Hour-of-day (24 buckets) conditioning (may be unstable)
This is typically unstable unless you have sufficient history per bucket.

In [ ]:
df["_hour"] = df[COL_T].dt.hour
buckets_24 = df["_hour"].to_numpy()

resid_all = (df[COL_Y] - df[COL_YHAT]).to_numpy()

Wb = 10  # per-hour bucket trailing points (tune with available data)
u24 = np.full(len(df), np.nan)

for t in range(len(df)):
    b = buckets_24[t]
    past_idx = np.where(buckets_24[:t] == b)[0]
    if len(past_idx) < Wb:
        continue
    past_idx = past_idx[-Wb:]
    window = resid_all[past_idx]
    r_t = resid_all[t]
    u24[t] = (np.sum(window < r_t) + 0.5 * np.sum(window == r_t)) / (len(window) + 1.0)

u24 = u24[~np.isnan(u24)]
print("24-bucket conditioned PIT length:", len(u24), "Wb=", Wb)

In [ ]:
plt.figure()
plt.hist(u24, bins=20, range=(0,1))
plt.title("PIT histogram (24-bucket hour-conditioned residual ECDF)")
plt.xlabel("u")
plt.ylabel("count")
plt.show()

def autocorr(x, lag):
    x = x - x.mean()
    return np.corrcoef(x[:-lag], x[lag:])[0,1]

max_lag = min(50, len(u24)-1)
acs24 = [autocorr(u24, k) for k in range(1, max_lag+1)]

plt.figure()
plt.plot(range(1, max_lag+1), acs24)
plt.title("PIT autocorrelation (24-bucket hour-conditioned)")
plt.xlabel("Lag")
plt.ylabel("ACF")
plt.show()

### 5.2 Coarse time-of-day (4 buckets) conditioning
More stable than 24 buckets under limited history.

In [ ]:
# --- 4 coarse time-of-day buckets ---
hour = df[COL_T].dt.hour
bucket4 = pd.cut(hour, bins=[-1, 5, 11, 17, 23], labels=[0, 1, 2, 3]).astype(int).to_numpy()

resid_all = (df[COL_Y] - df[COL_YHAT]).to_numpy()
y_all = df[COL_Y].to_numpy()
yhat_all = df[COL_YHAT].to_numpy()
t_all = df[COL_T].to_numpy()

Wb = 40  # trailing past points within same bucket

# --- PIT (past-only ECDF within bucket) ---
u4 = np.full(len(df), np.nan, dtype=float)

for t in range(len(df)):
    b = bucket4[t]
    past_idx = np.where(bucket4[:t] == b)[0]   # past only
    if len(past_idx) < Wb:
        continue
    past_idx = past_idx[-Wb:]
    window = resid_all[past_idx]
    r_t = resid_all[t]
    u4[t] = (np.sum(window < r_t) + 0.5 * np.sum(window == r_t)) / (len(window) + 1.0)

u4 = u4[~np.isnan(u4)]
print("4-bucket PIT length:", len(u4), "(Wb=", Wb, ")")

In [ ]:
plt.figure()
plt.hist(u4, bins=20, range=(0, 1))
plt.title("PIT histogram (4-bucket conditioned residual ECDF)")
plt.xlabel("u")
plt.ylabel("count")
plt.show()

def autocorr(x, lag):
    x = x - x.mean()
    return np.corrcoef(x[:-lag], x[lag:])[0, 1]

max_lag = min(50, len(u4) - 1)
acs4 = [autocorr(u4, k) for k in range(1, max_lag + 1)]

plt.figure()
plt.plot(range(1, max_lag + 1), acs4)
plt.title("PIT autocorrelation (4-bucket conditioned)")
plt.xlabel("Lag")
plt.ylabel("ACF")
plt.show()

In [ ]:
# --- Coverage using bucket-conditional residual quantiles (past-only) ---
for alpha in [0.2, 0.1]:
    covered = []
    for t in range(len(df)):
        b = bucket4[t]
        past_idx = np.where(bucket4[:t] == b)[0]   # past only
        if len(past_idx) < Wb:
            continue
        past_idx = past_idx[-Wb:]
        window = resid_all[past_idx]

        lo_r = np.quantile(window, alpha / 2)
        hi_r = np.quantile(window, 1 - alpha / 2)

        lo = yhat_all[t] + lo_r
        hi = yhat_all[t] + hi_r

        covered.append((y_all[t] >= lo) & (y_all[t] <= hi))

    n_eval = len(covered)
    if n_eval == 0:
        print(f"4-bucket coverage: no evaluable points (Wb={Wb}). Reduce Wb or use more data.")
    else:
        print(f"4-bucket — nominal {int((1-alpha)*100)}%: empirical {np.mean(covered):.3f} (n={n_eval})")

## 6. Regularized 4-bucket: bucket + global shrinkage + bias correction

Idea: bucket-conditional residuals can be noisy. We stabilize them by blending:

- **Bucket window** (same coarse time-of-day bucket) of size `Wb`
- **Global window** (most recent residuals regardless of bucket) of size `Wg`

We also do a simple **bias correction** by re-centering the blended residual window before taking quantiles.

In [ ]:
# --- Regularized hybrid window (bucket + global), with bias correction and a scale proxy ---
alphas = [0.2, 0.1]
Wb = 40   # bucket window size
Wg = 120  # global window size (increase if you have long history)

# store aligned series once
y_list, yhat_list, t_list = [], [], []
scale_list = []

# store base intervals per alpha
lo_map = {a: [] for a in alphas}
hi_map = {a: [] for a in alphas}
cov_map = {a: [] for a in alphas}

for t_idx in range(len(df)):
    b = bucket4[t_idx]

    past_b = np.where(bucket4[:t_idx] == b)[0]
    if len(past_b) < Wb or t_idx < Wg:
        continue

    past_b = past_b[-Wb:]
    win_b = resid_all[past_b]
    win_g = resid_all[t_idx-Wg:t_idx]

    window = np.concatenate([win_b, win_g])

    # bias correction: de-mean the blended window, but track mean for quantiles
    m = window.mean()
    window_c = window - m

    # ---- scale proxy (for potential scaled conformal later) ----
    med = np.median(window_c)
    mad = np.median(np.abs(window_c - med))
    scale_t = 1.4826 * mad
    if not np.isfinite(scale_t) or scale_t <= 1e-12:
        scale_t = float(np.std(window_c, ddof=1))
        if not np.isfinite(scale_t) or scale_t <= 1e-12:
            scale_t = 1.0

    # store the base intervals for EACH alpha
    for alpha in alphas:
        lo_r = np.quantile(window_c, alpha/2) + m
        hi_r = np.quantile(window_c, 1-alpha/2) + m

        lo = yhat_all[t_idx] + lo_r
        hi = yhat_all[t_idx] + hi_r

        lo_map[alpha].append(lo)
        hi_map[alpha].append(hi)
        cov_map[alpha].append((y_all[t_idx] >= lo) & (y_all[t_idx] <= hi))

    # store the aligned y/yhat/t ONCE
    y_list.append(y_all[t_idx])
    yhat_list.append(yhat_all[t_idx])
    t_list.append(t_all[t_idx])
    scale_list.append(scale_t)

y = np.asarray(y_list, dtype=float)
yhat = np.asarray(yhat_list, dtype=float)
t_eval = np.asarray(t_list, dtype=object)
scale = np.asarray(scale_list, dtype=float)

lo_80 = np.asarray(lo_map[0.2], dtype=float)
hi_80 = np.asarray(hi_map[0.2], dtype=float)
lo_90 = np.asarray(lo_map[0.1], dtype=float)
hi_90 = np.asarray(hi_map[0.1], dtype=float)

print("Aligned length:", len(y))
print("80% coverage:", float(np.mean(cov_map[0.2])))
print("90% coverage:", float(np.mean(cov_map[0.1])))
print("Scale summary (min/med/max):", float(np.min(scale)), float(np.median(scale)), float(np.max(scale)))

assert len(y) == len(scale) == len(lo_80) == len(hi_80) == len(lo_90) == len(hi_90) == len(t_eval)

## 7. Feasibility Assessment: Summary (write-up)

Use/update these bullets after you run the notebook on the 90-day sample and paste in the realized numbers.

### Interpretation
- Parametric Gaussian + rolling σ often under-covers (under-dispersion) and produces dependent PIT.
- Rolling empirical residuals improves marginal calibration but may still show dependence.
- Fine-grained conditioning (24 buckets) requires a lot of history per bucket; otherwise quantiles are unstable.
- Coarse conditioning (4 buckets) is more stable.
- Regularization via bucket+global blending + bias correction can achieve conservative or near-nominal coverage even under limited history.

### Feasibility conclusion
> ENTSOG point forecasts can be feasibly transformed into predictive distributions suitable for probabilistic evaluation, but calibration depends on assumptions and regularization.

## 8. Save aligned artifacts (derived_long)

This notebook is intended to save into `data/derived_long/` (matching your screenshot).

In [ ]:
OUT = REPO_ROOT / "data" / "derived_long"
OUT.mkdir(parents=True, exist_ok=True)

np.save(OUT / "entsog_y.npy", y)
np.save(OUT / "entsog_yhat.npy", yhat)
np.save(OUT / "entsog_t.npy", t_eval)

np.save(OUT / "entsog_lo_base_80.npy", lo_80)
np.save(OUT / "entsog_hi_base_80.npy", hi_80)

np.save(OUT / "entsog_lo_base_90.npy", lo_90)
np.save(OUT / "entsog_hi_base_90.npy", hi_90)

# optional: a scale proxy for later scaled conformal
np.save(OUT / "entsog_scale.npy", scale)

print("Saved aligned artifacts to:", OUT)